In [61]:

import pandas as pd

df_full = pd.read_csv("../../data/1_derived/2_sf_properties_for_geocoding_standardized.csv")
print(f"Full shape: {df_full.shape}")

# Load existing geocoded results and identify rows that still need geocoding
existing_path = "../../data/1_derived/3_sf_properties_census_geocoded.csv"
df_existing = pd.read_csv(existing_path)

# FIX: use PropertyId (not positional index) to identify already-matched rows
matched_ids = set(
    df_existing.loc[df_existing["match"] == "Match", "PropertyId"].dropna()
)
rematch_mask = ~df_full["PropertyId"].isin(matched_ids)
print(f"Already matched: {(~rematch_mask).sum()} | To re-geocode: {rematch_mask.sum()}")

# Subset to only unmatched rows
df = df_full[rematch_mask].copy()
print(f"Subset for geocoding: {df.shape}")
df.head()


Full shape: (9352, 11)
Already matched: 8980 | To re-geocode: 372
Subset for geocoding: (372, 11)


,LeaseCompId,PropertyId,PropertyName,Address.DeliveryAddress,Address.Locality,Address.Subdivision,Address.PostalCode,Address.County,Market,Submarket,full_address_std
6,70054428.0,36162.0,NaN,710 Market St,San Francisco,CA,94102.0,San Francisco,San Francisco,Union Square,"710 MARKET ST, SAN FRANCISCO, CA 94102, UNITED..."
27,170067741.0,36889.0,Genesis SSF - North Tower,2 Tower Pl,South San Francisco,CA,94080.0,San Mateo,San Francisco,South San Francisco,"2 TOWER PL, SOUTH SAN FRANCISCO, CA 94080, UNI..."
52,126883611.0,47879.0,NaN,1405 Huntington Ave,South San Francisco,CA,94080.0,San Mateo,San Francisco,South San Francisco,"1405 HUNTINGTON AVE, SOUTH SAN FRANCISCO, CA 9..."
77,10373953.0,52103.0,NaN,690 Market St,San Francisco,CA,94104.0,San Francisco,San Francisco,Financial District,"690 MARKET ST, SAN FRANCISCO, CA 94104, UNITED..."
86,70301420.0,53542.0,Pier 29,Pier 29,San Francisco,CA,94121.0,San Francisco,San Francisco,Waterfront/North Beach,"PIER 29, SAN FRANCISCO, CA 94121, UNITED STATES"


In [62]:


# --- Step 1: Parse full_address_std into Census batch format ---
# Expected format: "STREET, CITY, ST ZIP, UNITED STATES"

def parse_address(addr):
    """Split 'STREET, CITY, STATE ZIP, COUNTRY' into Census batch components."""
    parts = [p.strip() for p in str(addr).split(',')]
    street   = parts[0] if len(parts) > 0 else ''
    city     = parts[1] if len(parts) > 1 else ''
    state_zip = parts[2].strip() if len(parts) > 2 else ''
    tokens   = state_zip.split()
    state    = tokens[0] if len(tokens) > 0 else ''
    zip_code = tokens[1] if len(tokens) > 1 else ''
    return street, city, state, zip_code

parsed = df['full_address_std'].apply(parse_address)
batch_input = pd.DataFrame(parsed.tolist(), columns=['street', 'city', 'state', 'zip'])
batch_input.insert(0, 'id', df['PropertyId'].astype(int).values)  # use PropertyId as batch ID

print(f"Batch input shape: {batch_input.shape}")
batch_input.head()



Batch input shape: (372, 5)


,id,street,city,state,zip
0,36162,710 MARKET ST,SAN FRANCISCO,CA,94102
1,36889,2 TOWER PL,SOUTH SAN FRANCISCO,CA,94080
2,47879,1405 HUNTINGTON AVE,SOUTH SAN FRANCISCO,CA,94080
3,52103,690 MARKET ST,SAN FRANCISCO,CA,94104
4,53542,PIER 29,SAN FRANCISCO,CA,94121


In [63]:

# --- Step 2: Batch geocode via Census Geocoder ---
# Submits addresses in chunks of 500 (API limit: 10,000; smaller chunks reduce timeout risk)
import requests
import io
import time

CENSUS_BATCH_URL = "https://geocoding.geo.census.gov/geocoder/geographies/addressbatch"
CENSUS_SINGLE_URL = "https://geocoding.geo.census.gov/geocoder/geographies/address"
CHUNK_SIZE = 500

def geocode_chunk(chunk_df, retries=3):
    """POST a chunk to the Census batch geocoder; returns raw response text."""
    csv_bytes = chunk_df.to_csv(index=False, header=False).encode("utf-8")
    for attempt in range(1, retries + 1):
        try:
            resp = requests.post(
                CENSUS_BATCH_URL,
                files={"addressFile": ("addresses.csv", csv_bytes, "text/csv")},
                data={"benchmark": "Public_AR_Current", "vintage": "Current_Current"},
                timeout=300,
            )
            resp.raise_for_status()
            return resp.text
        except Exception as e:
            print(f"  Attempt {attempt} failed: {e}")
            if attempt < retries:
                time.sleep(5)
    return ""

def parse_census_response(raw_text):
    """Parse Census batch response CSV into a DataFrame."""
    cols = [
        "id", "input_address", "match", "match_type",
        "matched_address", "coordinates",
        "tiger_line_id", "side",
        "state_fips", "county_fips", "census_tract", "census_block",
    ]
    try:
        return pd.read_csv(
            io.StringIO(raw_text),
            names=cols,
            dtype={"state_fips": str, "county_fips": str,
                   "census_tract": str, "census_block": str},
        )
    except Exception:
        return pd.DataFrame(columns=cols)

def resolve_tie_single(row_id, street, city, state, zipcode, retries=3):
    """Query the single-address API to get all candidates for a Tie address.
    The batch API returns no coordinates for Ties; the single-address API does."""
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(
                CENSUS_SINGLE_URL,
                params={
                    "street": street, "city": city,
                    "state": state, "zip": zipcode,
                    "benchmark": "Public_AR_Current",
                    "vintage": "Current_Current",
                    "format": "json",
                },
                timeout=60,
            )
            resp.raise_for_status()
            data = resp.json()
            matches = data.get("result", {}).get("addressMatches", [])
            if not matches:
                return []

            candidates = []
            for m in matches:
                coords = m.get("coordinates", {})
                geo = m.get("geographies", {})
                blocks = geo.get("2020 Census Blocks", [])
                blk = blocks[0] if blocks else {}
                candidates.append({
                    "id": row_id,
                    "match": "Tie",
                    "match_type": "Tie",
                    "matched_address": m.get("matchedAddress", ""),
                    "coordinates": f"{coords.get('x', '')},{coords.get('y', '')}",
                    "state_fips": str(blk.get("STATE", "")),
                    "county_fips": str(blk.get("COUNTY", "")),
                    "census_tract": str(blk.get("TRACT", "")),
                    "census_block": str(blk.get("BLOCK", "")),
                })
            return candidates
        except Exception as e:
            print(f"  Tie resolve attempt {attempt} for id={row_id} failed: {e}")
            if attempt < retries:
                time.sleep(2)
    return []

# --- Run batch geocoding ---
results = []
total_chunks = (len(batch_input) + CHUNK_SIZE - 1) // CHUNK_SIZE

for i in range(total_chunks):
    chunk = batch_input.iloc[i * CHUNK_SIZE : (i + 1) * CHUNK_SIZE]
    print(f"Geocoding chunk {i+1}/{total_chunks} ({len(chunk)} records)...", end=" ")
    raw = geocode_chunk(chunk)
    parsed_chunk = parse_census_response(raw)
    results.append(parsed_chunk)
    print(f"got {len(parsed_chunk)} rows back.")

geo_df = pd.concat(results, ignore_index=True)
print(f"\nTotal geocoded rows: {len(geo_df)}")
print(geo_df["match"].value_counts())

# --- Resolve Tie rows via single-address API ---
# The batch API returns only (id, address, "Tie") with no coordinates or FIPS.
tie_ids = geo_df.loc[geo_df["match"] == "Tie", "id"].unique()
if len(tie_ids) > 0:
    print(f"\nResolving {len(tie_ids)} Tie addresses via single-address API...")
    # Build a lookup from batch_input so we can pass street/city/state/zip
    tie_input = batch_input[batch_input["id"].isin(tie_ids)].set_index("id")

    resolved_rows = []
    for idx, tid in enumerate(tie_ids):
        row = tie_input.loc[tid]
        candidates = resolve_tie_single(tid, row["street"], row["city"], row["state"], row["zip"])
        resolved_rows.extend(candidates)
        if (idx + 1) % 10 == 0 or idx + 1 == len(tie_ids):
            print(f"  Resolved {idx + 1}/{len(tie_ids)} ties ({len(resolved_rows)} candidates so far)")
        time.sleep(0.25)  # be polite to the API

    if resolved_rows:
        resolved_df = pd.DataFrame(resolved_rows)
        # Replace the skeleton Tie rows with the fully resolved ones
        geo_df = geo_df[geo_df["match"] != "Tie"]
        geo_df = pd.concat([geo_df, resolved_df], ignore_index=True)
        print(f"Replaced Tie skeletons with {len(resolved_df)} resolved rows")

print(f"\nFinal geocoded rows: {len(geo_df)}")
print(geo_df["match"].value_counts())
geo_df.head()


Geocoding chunk 1/1 (372 records)... got 372 rows back.

Total geocoded rows: 372
match
No_Match    353
Tie          19
Name: count, dtype: int64

Resolving 19 Tie addresses via single-address API...
  Resolved 10/19 ties (18 candidates so far)
  Resolved 19/19 ties (36 candidates so far)
Replaced Tie skeletons with 36 resolved rows

Final geocoded rows: 389
match
No_Match    353
Tie          36
Name: count, dtype: int64


,id,input_address,match,match_type,matched_address,coordinates,tiger_line_id,side,state_fips,county_fips,census_tract,census_block
0,324446,"390 SELBY ST, SAN FRANCISCO, CA, 94124",No_Match,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,113778,"1600 SIERRA POINT PKY, BRISBANE, CA, 94005",No_Match,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5521120,"AVENUE OF THE PALMS, SAN FRANCISCO, CA, 94130",No_Match,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,896929,"1011 EL CAMINO REAL, MENLO PARK, CA, 94025",No_Match,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6678870,"1034 EL CAMINO REAL, REDWOOD CITY, CA, 94063",No_Match,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [64]:


# --- Step 3: Extract lat/lon, build FIPS, and merge back to df_full via PropertyId ---

def split_coords(coord_str):
    """Return (longitude, latitude) from 'lon,lat' string."""
    try:
        lon, lat = str(coord_str).split(",")
        return float(lon.strip()), float(lat.strip())
    except Exception:
        return None, None

geo_df[["longitude", "latitude"]] = pd.DataFrame(
    geo_df["coordinates"].apply(split_coords).tolist(),
    index=geo_df.index,
)

# Pad FIPS codes to standard widths and build full 15-digit GEOID
geo_df["state_fips"]   = geo_df["state_fips"].str.zfill(2)
geo_df["county_fips"]  = geo_df["county_fips"].str.zfill(3)
geo_df["census_tract"] = geo_df["census_tract"].str.zfill(6)
geo_df["census_block"] = geo_df["census_block"].str.zfill(4)

geo_df["geoid"] = (
    geo_df["state_fips"]
    + geo_df["county_fips"]
    + geo_df["census_tract"]
    + geo_df["census_block"]
)

# For "Tie" rows the Census API returns multiple candidates per ID.
# Aggregate all tie candidates into pipe-delimited columns so no data is lost.
tie_mask = geo_df["match"] == "Tie"
if tie_mask.any():
    tie_agg = (
        geo_df[tie_mask]
        .groupby("id", sort=False)
        .agg(
            tie_matched_addresses=("matched_address", lambda x: " | ".join(x.fillna("").astype(str))),
            tie_latitudes        =("latitude",        lambda x: " | ".join(x.dropna().astype(str))),
            tie_longitudes       =("longitude",       lambda x: " | ".join(x.dropna().astype(str))),
            tie_geoids           =("geoid",           lambda x: " | ".join(x.fillna("").astype(str))),
        )
        .reset_index()
    )
    print(f"Tie records: {tie_mask.sum()} rows across {tie_agg.shape[0]} unique IDs")
else:
    tie_agg = pd.DataFrame(
        columns=["id", "tie_matched_addresses", "tie_latitudes", "tie_longitudes", "tie_geoids"]
    )

# Deduplicate to one row per ID (for ties, keep the first candidate as the primary values)
geo_df_dedup = geo_df.drop_duplicates(subset="id", keep="first").copy()
geo_df_dedup["id"] = pd.to_numeric(geo_df_dedup["id"], errors="coerce")

# Attach all tie candidate data as extra columns
tie_agg["id"] = pd.to_numeric(tie_agg["id"], errors="coerce")
geo_df_dedup = geo_df_dedup.merge(tie_agg, on="id", how="left")

# 'id' is PropertyId — rename for the merge
geocode_cols = [
    "match", "matched_address",
    "latitude", "longitude",
    "state_fips", "county_fips", "census_tract", "census_block", "geoid",
    "tie_matched_addresses", "tie_latitudes", "tie_longitudes", "tie_geoids",
]
new_geo = geo_df_dedup[["id"] + geocode_cols].rename(columns={"id": "PropertyId"})

# Bring in previously-matched geocode data (keyed by PropertyId, not position)
existing_geo_cols = [c for c in geocode_cols if c in df_existing.columns]
prev_matched = (
    df_existing.loc[df_existing["match"] == "Match"]
    .drop_duplicates(subset="PropertyId", keep="first")
    [["PropertyId"] + existing_geo_cols]
)

# Combine: new results + previously-matched results (new results take priority)
all_geo = pd.concat([new_geo, prev_matched], ignore_index=True).drop_duplicates(
    subset="PropertyId", keep="first"
)

# Always rebuild from df_full to avoid stale/scrambled property data
df_geo = df_full.merge(all_geo, on="PropertyId", how="left")

print(f"Final shape: {df_geo.shape}")
print(f"Unique PropertyIds: {df_geo['PropertyId'].nunique()} / {len(df_geo)}")
print(f"Matched: {(df_geo['match'] == 'Match').sum()} / {len(df_geo)}")
print(f"Tied:    {(df_geo['match'] == 'Tie').sum()} / {len(df_geo)}")

df_geo[["PropertyId", "full_address_std", "match", "latitude", "longitude",
        "geoid", "tie_matched_addresses", "tie_latitudes", "tie_longitudes"]].head(10)



Tie records: 36 rows across 18 unique IDs
Final shape: (9352, 24)
Unique PropertyIds: 9352 / 9352
Matched: 8980 / 9352
Tied:    18 / 9352


,PropertyId,full_address_std,match,latitude,longitude,geoid,tie_matched_addresses,tie_latitudes,tie_longitudes
0,36156.0,"445 BUSH ST, SAN FRANCISCO, CA 94108, UNITED S...",Match,37.790605,-122.404826,60750117004006.0,NaN,NaN,NaN
1,36157.0,"33 NEW MONTGOMERY ST, SAN FRANCISCO, CA 94105,...",Match,37.788386,-122.401551,60750615013000.0,NaN,NaN,NaN
2,36158.0,"115 SANSOME ST, SAN FRANCISCO, CA 94104, UNITE...",Match,37.791340,-122.400850,60750117002018.0,NaN,NaN,NaN
3,36159.0,"501 2ND ST, SAN FRANCISCO, CA 94107, UNITED ST...",Match,37.782822,-122.393193,60750615083001.0,NaN,NaN,NaN
4,36160.0,"110 SUTTER ST, SAN FRANCISCO, CA 94104, UNITED...",Match,37.790032,-122.402722,60750117004003.0,NaN,NaN,NaN
5,36161.0,"1033 MARKET ST, SAN FRANCISCO, CA 94103, UNITE...",Match,37.781543,-122.410915,60750176043000.0,NaN,NaN,NaN
6,36162.0,"710 MARKET ST, SAN FRANCISCO, CA 94102, UNITED...",No_Match,NaN,NaN,NaN,NaN,NaN,NaN
7,36163.0,"327 GRANT AVE, SAN FRANCISCO, CA 94108, UNITED...",Match,37.790124,-122.405583,60750117004007.0,NaN,NaN,NaN
8,36164.0,"50 MENDELL ST, SAN FRANCISCO, CA 94124, UNITED...",Match,37.743513,-122.383498,60759809001023.0,NaN,NaN,NaN
9,36165.0,"1155 MISSION ST, SAN FRANCISCO, CA 94103, UNIT...",Match,37.778224,-122.412088,60750176032000.0,NaN,NaN,NaN


In [65]:

# --- Step 4: Save geocoded output ---
import os

out_path = "../../data/1_derived/3_sf_properties_census_geocoded.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df_geo.to_csv(out_path, index=False)
print(f"Saved {len(df_geo)} rows to {out_path}")


Saved 9352 rows to ../../data/1_derived/3_sf_properties_census_geocoded.csv
